# v7 — Ablação de modalidades + sensibilidade de hiperparâmetros

Responde às duas perguntas do orientador, **no mesmo protocolo do notebook 09** (5-fold estratificado sobre os 1011 casos do Derm7pt, `random_state=42`, cabeça treinada em 4/5 Derm7pt + HAM cortado, 25 épocas fixas, AdamW + Cosine, sem seleção no teste):

**Parte A — Ablação de modalidades.** A arquitetura multimodal é necessária?
- `multimodal` (v7): imagem + metadados
- `image`: só o ramo visual (ramo tabular removido)
- `tabular`: só os metadados (sexo + localização)
- **Permutação dos metadados** no teste do modelo multimodal (quanto o modelo *usa* os metadados)
- Baseline de classe majoritária

**Parte B — Sensibilidade a hiperparâmetros (one-at-a-time).** Partindo da configuração do v7, varia-se **um hiperparâmetro por vez** (LR, γ da Focal Loss, ponderação de classes, label smoothing, épocas, corte de NEV). É uma **análise de sensibilidade**, não uma busca: a configuração final NÃO é trocada com base nesses números (escolher pelo teste seria vazamento).

Cada configuração roda com **3 sementes × 5 dobras = 15 execuções**. Comparações vs v7 usam teste de Wilcoxon pareado (mesma dobra e semente).

Tempo estimado (T4): ~60–90 min para cachear embeddings (só na 1ª vez; o cache é salvo em `/kaggle/working`) + ~30–60 min de treinos.

In [ ]:
import os, sys, site, importlib, subprocess
REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "huggingface_hub"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches(); site.main()
print("Setup OK")

In [ ]:
import time, json
import pandas as pd
from collections import Counter
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedKFold
from kaggle_secrets import UserSecretsClient

from melanoma_tcc.data.preprocessing import (
    Derm7ptUnifiedDataset, HAM10000Dataset, classification_collate_fn,
    METADATA_DIM_V5, HAM_DX_TO_GROUP, ham10000_train_val_split,
)
from melanoma_tcc.model.classifier import build_dermclassifier
from melanoma_tcc.model.losses import FocalLoss, compute_class_weights

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7pt-multimodal/release_v0"
DERM_META = f"{DERM7PT_DIR}/meta/meta.csv"
DERM_IMAGES = f"{DERM7PT_DIR}/images"
HAM_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
HAM_META = f"{HAM_DIR}/HAM10000_metadata.csv"
OUT_DIR = "/kaggle/working/v7-ablacao"
CACHE = "/kaggle/working/emb_cache_v7.pt"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ===== Nucleo: cabecas (multimodal / so imagem / so metadados) + treino + avaliacao =====
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, recall_score, precision_score

VISION_DIM, META_DIM, HIDDEN, N_CLASSES, MEL_LABEL = 1152, 10, 256, 5, 2


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class Head(nn.Module):
    """Mesma cabeca do DermClassifier (v7). mode: 'multimodal' | 'image' | 'tabular'.
    Nas variantes unimodais o ramo ausente eh removido (nao zerado), e o classificador
    recebe 256-d em vez de 512-d. Todo o resto (camadas, dropout, init) eh identico."""

    def __init__(self, mode="multimodal", dropout_head=0.2):
        super().__init__()
        self.mode = mode
        if mode in ("multimodal", "image"):
            self.vision_proj = nn.Sequential(
                nn.LayerNorm(VISION_DIM), nn.Linear(VISION_DIM, HIDDEN), nn.GELU(), nn.Dropout(0.1))
        if mode in ("multimodal", "tabular"):
            self.metadata_encoder = nn.Sequential(
                nn.Linear(META_DIM, 128), nn.GELU(), nn.Linear(128, HIDDEN), nn.GELU(), nn.Dropout(0.1))
        in_dim = HIDDEN * (2 if mode == "multimodal" else 1)
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, HIDDEN), nn.GELU(), nn.Dropout(dropout_head), nn.Linear(HIDDEN, N_CLASSES))

    def forward(self, emb, md):
        feats = []
        if self.mode in ("multimodal", "image"):
            feats.append(self.vision_proj(emb))
        if self.mode in ("multimodal", "tabular"):
            feats.append(self.metadata_encoder(md))
        return self.classifier(torch.cat(feats, dim=-1))


def train_head(tr_emb, tr_md, tr_lb, cfg, seed, device):
    """Mesmo protocolo do notebook 09: AdamW + Cosine, epocas fixas, sem selecao no teste."""
    set_seed(seed)
    model = Head(cfg["mode"], cfg.get("dropout_head", 0.2)).to(device)
    counts = [max(1, int((tr_lb == i).sum())) for i in range(N_CLASSES)]
    alpha = None if cfg["weights"] == "none" else compute_class_weights(counts, mode=cfg["weights"])
    criterion = FocalLoss(alpha=alpha, gamma=cfg["gamma"], label_smoothing=cfg["smoothing"])
    opt = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = CosineAnnealingLR(opt, T_max=cfg["epochs"])
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(tr_emb, tr_md, tr_lb), batch_size=cfg["batch"], shuffle=True, generator=g)
    for _ in range(cfg["epochs"]):
        model.train()
        for emb, md, lb in loader:
            emb, md, lb = emb.to(device), md.float().to(device), lb.to(device)
            loss = criterion(model(emb, md), lb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
    return model


@torch.no_grad()
def predict(model, emb, md, device):
    model.eval()
    return F.softmax(model(emb.to(device), md.float().to(device)).float(), dim=-1).argmax(-1).cpu().numpy()


def metrics(y, p):
    return {
        "acc": float((y == p).mean()),
        "macro_f1": float(f1_score(y, p, average="macro", labels=list(range(N_CLASSES)), zero_division=0)),
        "mel_rec": float(recall_score(y, p, labels=[MEL_LABEL], average="macro", zero_division=0)),
        "mel_prec": float(precision_score(y, p, labels=[MEL_LABEL], average="macro", zero_division=0)),
        **{f"f1_{c}": float(v) for c, v in zip(
            ["BCC", "NEV", "MEL", "SK", "MISC"],
            f1_score(y, p, average=None, labels=list(range(N_CLASSES)), zero_division=0))},
    }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert METADATA_DIM_V5 == META_DIM
for m in ("multimodal", "image", "tabular"):
    print(f"{m:10s}: {sum(p.numel() for p in Head(m).parameters()):,} params treinaveis")

In [ ]:
# ===== Embeddings: Derm7pt (1011) + HAM train COMPLETO (o corte de NEV eh feito por indice) =====
ham_train_df, _ = ham10000_train_val_split(HAM_META, val_ratio=0.15, seed=42, filter_unknown=True)
ham_train_df = ham_train_df.reset_index(drop=True)
ham_train_df['group'] = ham_train_df['dx'].str.strip().str.lower().map(HAM_DX_TO_GROUP).fillna('MISC')

if os.path.exists(CACHE):
    c = torch.load(CACHE)
    derm_emb, derm_md, derm_lb = c['derm']
    ham_emb, ham_md, ham_lb = c['ham']
    print("Cache carregado.")
else:
    clf, processor = build_dermclassifier(hf_token=UserSecretsClient().get_secret('mellanoma_TCC'),
                                          num_classes=5, metadata_dim=METADATA_DIM_V5, freeze_vision=True)
    clf = clf.to(device); clf.vision_encoder = clf.vision_encoder.to(device); clf.eval()

    @torch.no_grad()
    def encode_dataset(ds, bs=32):      # identico ao notebook 09
        loader = DataLoader(ds, batch_size=bs, shuffle=False,
                            collate_fn=classification_collate_fn, num_workers=4)
        E, M, L = [], [], []
        t0 = time.time()
        for i, b in enumerate(loader):
            E.append(clf.encode_image(b['pixel_values'].to(device)).cpu())
            M.append(b['metadata']); L.append(b['labels'])
            if i % 50 == 0:
                print(f"  {i*bs}/{len(ds)}  ({time.time()-t0:.0f}s)")
        return torch.cat(E), torch.cat(M), torch.cat(L)

    derm_full = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor, indexes_csv=None, augment=False)
    ham_train = HAM10000Dataset(ham_train_df, HAM_DIR, processor, augment=False)
    assert len(ham_train) == len(ham_train_df)
    print("Cacheando Derm7pt..."); derm_emb, derm_md, derm_lb = encode_dataset(derm_full)
    print("Cacheando HAM train completo..."); ham_emb, ham_md, ham_lb = encode_dataset(ham_train)
    torch.save({'derm': (derm_emb, derm_md, derm_lb), 'ham': (ham_emb, ham_md, ham_lb)}, CACHE)
    del clf; torch.cuda.empty_cache()
    print(f"Cache salvo em {CACHE} (baixe-o para nao recomputar numa proxima sessao)")

derm_y = derm_lb.numpy()
print("Derm7pt:", tuple(derm_emb.shape), "| HAM train:", tuple(ham_emb.shape))
print("HAM dist:", dict(Counter(ham_train_df['group'])))

In [ ]:
def ham_subset(nev_target):
    """Indices do HAM train apos cortar NEV para nev_target (mesma amostragem do v7: random_state=42)."""
    nev_mask = (ham_train_df['group'] == 'NEV').values
    if nev_target is None or nev_mask.sum() <= nev_target:
        return torch.arange(len(ham_train_df))
    keep_nev = ham_train_df[nev_mask].sample(n=nev_target, random_state=42).index.values
    idx = np.sort(np.concatenate([keep_nev, np.where(~nev_mask)[0]]))
    return torch.as_tensor(idx)

BASE = dict(mode="multimodal", lr=5e-4, wd=0.01, gamma=2.0, smoothing=0.05,
            weights="inverse_sqrt", epochs=25, batch=32, nev_target=1400)
SEEDS = [42, 7, 2024]
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
             .split(np.zeros(len(derm_y)), derm_y))          # mesmas dobras do notebook 09

def run_cv(name, cfg, permute_md=False):
    rows = []
    h_idx = ham_subset(cfg['nev_target'])
    he, hm, hl = ham_emb[h_idx], ham_md[h_idx], ham_lb[h_idx]
    for seed in SEEDS:
        for k, (tr, te) in enumerate(FOLDS, 1):
            model = train_head(torch.cat([derm_emb[tr], he]), torch.cat([derm_md[tr], hm]),
                               torch.cat([derm_lb[tr], hl]), cfg, seed, device)
            y = derm_y[te]
            rows.append({'config': name, 'seed': seed, 'fold': k, 'eval': 'normal',
                         **metrics(y, predict(model, derm_emb[te], derm_md[te], device))})
            if permute_md:   # embaralha os metadados entre os casos de teste (imagem intacta)
                g = torch.Generator().manual_seed(seed * 100 + k)
                perm = torch.randperm(len(te), generator=g)
                rows.append({'config': name, 'seed': seed, 'fold': k, 'eval': 'md_permutado',
                             **metrics(y, predict(model, derm_emb[te], derm_md[te][perm], device))})
    return rows

def summarize(df, cols=('acc', 'macro_f1', 'mel_rec', 'mel_prec')):
    """media sobre 15 execucoes; desvio entre as 5 dobras (apos media das sementes), como no nb 09."""
    out = {}
    for c in cols:
        per_fold = df.groupby('fold')[c].mean()
        out[c] = f"{df[c].mean():.3f} ± {per_fold.std(ddof=0):.3f}"
    return out

def paired(df_a, df_b, col):
    a = df_a.sort_values(['seed', 'fold'])[col].values
    b = df_b.sort_values(['seed', 'fold'])[col].values
    d = a - b
    p = wilcoxon(a, b).pvalue if np.any(d != 0) else 1.0
    return d.mean(), p

## Parte A — Ablação de modalidades

In [ ]:
t0 = time.time()
rows_A = []
rows_A += run_cv('multimodal', BASE, permute_md=True)
rows_A += run_cv('image', {**BASE, 'mode': 'image'})
rows_A += run_cv('tabular', {**BASE, 'mode': 'tabular'})
A = pd.DataFrame(rows_A)
A.to_csv(f"{OUT_DIR}/ablacao_modalidades.csv", index=False)
print(f"Parte A concluida em {(time.time()-t0)/60:.1f} min")

In [ ]:
# Baseline de classe majoritaria (NEV) nas mesmas dobras
maj = [metrics(derm_y[te], np.full(len(te), 1)) for _, te in FOLDS]
print(f"Classe majoritaria (sempre NEV): acc = {np.mean([m['acc'] for m in maj]):.3f}, "
      f"macro-F1 = {np.mean([m['macro_f1'] for m in maj]):.3f}\n")

mm = A[(A.config == 'multimodal') & (A['eval'] == 'normal')]
variants = {
    'Multimodal (v7)':            mm,
    'Só imagem':                  A[A.config == 'image'],
    'Só metadados':               A[A.config == 'tabular'],
    'Multimodal, md permutados':  A[(A.config == 'multimodal') & (A['eval'] == 'md_permutado')],
}
tab = []
for name, d in variants.items():
    r = {'Configuração': name, **summarize(d)}
    if name != 'Multimodal (v7)':
        for c in ('acc', 'macro_f1', 'mel_rec'):
            diff, p = paired(mm, d, c)
            r[f'Δ{c} (v7 − esta)'] = f"{diff:+.3f} (p={p:.3f})"
    tab.append(r)
tab_A = pd.DataFrame(tab)
tab_A.to_csv(f"{OUT_DIR}/tabela_ablacao.csv", index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
print(tab_A.to_string(index=False))

print("\nF1 por classe (média das 15 execuções):")
print(pd.DataFrame({n: d[[f'f1_{c}' for c in ['BCC','NEV','MEL','SK','MISC']]].mean()
                    for n, d in variants.items()}).round(3).to_string())

## Parte B — Sensibilidade a hiperparâmetros (one-at-a-time)

In [ ]:
VARIACOES = {
    'lr=1e-4':          {'lr': 1e-4},
    'lr=1e-3':          {'lr': 1e-3},
    'gamma=0 (CE)':     {'gamma': 0.0},
    'gamma=1':          {'gamma': 1.0},
    'gamma=3':          {'gamma': 3.0},
    'pesos=nenhum':     {'weights': 'none'},
    'pesos=inverse':    {'weights': 'inverse'},
    'smoothing=0':      {'smoothing': 0.0},
    'smoothing=0.1':    {'smoothing': 0.1},
    'epocas=15':        {'epochs': 15},
    'epocas=40':        {'epochs': 40},
    'NEV=1000':         {'nev_target': 1000},
    'NEV=1800':         {'nev_target': 1800},
    'NEV=sem corte':    {'nev_target': None},
}
t0 = time.time()
rows_B = []
for name, change in VARIACOES.items():
    rows_B += run_cv(name, {**BASE, **change})
    print(f"  {name:16s} ok ({(time.time()-t0)/60:.1f} min)")
B = pd.DataFrame(rows_B)
B.to_csv(f"{OUT_DIR}/sensibilidade_hparams.csv", index=False)

In [ ]:
tab = [{'Configuração': 'v7 (referência)', **summarize(mm), 'Δmacro-F1 vs v7': '—', 'ΔMEL rec vs v7': '—'}]
for name in VARIACOES:
    d = B[B.config == name]
    df1, p1 = paired(d, mm, 'macro_f1')
    dmr, p2 = paired(d, mm, 'mel_rec')
    tab.append({'Configuração': name, **summarize(d),
                'Δmacro-F1 vs v7': f"{df1:+.3f} (p={p1:.3f})",
                'ΔMEL rec vs v7': f"{dmr:+.3f} (p={p2:.3f})"})
tab_B = pd.DataFrame(tab)
tab_B.to_csv(f"{OUT_DIR}/tabela_sensibilidade.csv", index=False)
print(tab_B.to_string(index=False))

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/v7-ablacao", "zip", OUT_DIR)
print("Baixe /kaggle/working/v7-ablacao.zip (CSVs) e mande junto o texto impresso das duas tabelas.")